# 胸部X線画像による肺炎分類

Kaggle **Chest X-Ray Images (Pneumonia)** の二値分類 (NORMAL / PNEUMONIA)。
再利用処理は `src/`(dataset/models/train/evaluate/gradcam)に分割し、本 Notebook から呼び出す。

比較する4モデル: **Baseline CNN / ResNet18(Frozen) / ResNet18(Fine-tuned) / EfficientNet-B0**
誤分類分析(方向別・確信度別)と **Grad-CAM** による着目領域の可視化まで行う。

> **注意:** 学習用データセットでの画像分類タスクの検証であり、**医療診断を目的としたものではありません。**

#### 以下①～④はGoogle Colabで実行する場合のみ実行

In [ ]:
## ① kaggle.jsonのアップロード
from google.colab import files
files.upload()

In [ ]:
## ② 配置と権限設定
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
## ③ ダウンロードと解凍
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

In [ ]:
## ④ GoogleドライブをColabに接続
from google.colab import drive
drive.mount('/content/drive')

### ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import random

import time
from PIL import Image
import torch
import torchvision
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## src モジュールの読み込み

In [ ]:
import sys
sys.path.append("../src")   # notebooks/ の隣の src/ を参照

from dataset import make_transforms, build_loaders
from models import (build_baseline, build_resnet18, build_efficientnet_b0,
                    freeze_backbone_resnet, unfreeze_resnet_later)
from train import train, plot_history, trainable_params
from evaluate import (evaluate, measure_speed, model_stats,
                      show_misclassified_by_type, show_confidence_cases)
from gradcam import show_gradcam, get_target_layer

### データのクリーンアップ（macOSのゴミ除去）

解凍時に `__MACOSX` フォルダや `._` で始まるファイルが混入すると、画像読み込みでエラーになる。先に削除しておく。

In [ ]:
!rm -rf /content/chest_xray/__MACOSX
!find /content/chest_xray -name '._*' -delete
print("cleanup done")

### trainデータの枚数確認

In [ ]:
# trainの正常と異常の枚数を確認
# Colabの場合のみ'/content/chest_xray'
DATA_DIR = '/content/chest_xray/'
train_dir = os.path.join(DATA_DIR, 'train')
val_dir = os.path.join(DATA_DIR, 'val')
test_dir = os.path.join(DATA_DIR, 'test')

train_normal_count = len(os.listdir(os.path.join(train_dir, 'NORMAL')))
train_pneumonia_count = len(os.listdir(os.path.join(train_dir, 'PNEUMONIA')))

print(f"訓練(NORMAL)の枚数: {train_normal_count}枚")
print(f"訓練(PNEUMONIA)の枚数: {train_pneumonia_count}枚")

### 各データセットのデータ枚数とクラス比率

In [ ]:
# train/validation/testの枚数とクラス比率
for split, path in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    print(f"\n--- {split} ---")
    counts = {}
    total = 0
    for label in ['NORMAL', 'PNEUMONIA']:
      # os.path.join：パスとフォルダ名を結合してフルパスを作成
      # os.listdir：ファイル名の文字列リスト
        n = len(os.listdir(os.path.join(path, label)))
        counts[label] = n
        total += n
    for label, n in counts.items():
        print(f'{label}: {n}枚 ({n/total*100:.1f}%)')
    print(f'合計: {total}枚')

In [ ]:
counts_all = {}

for split, path in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    counts = {}
    for label in ['NORMAL', 'PNEUMONIA']:
        n = len(os.listdir(os.path.join(path, label)))
        counts[label] = n
    counts_all[split] = counts


splits = ['train', 'val', 'test']
normal_counts = [counts_all[s]['NORMAL'] for s in splits]
pneumonia_counts = [counts_all[s]['PNEUMONIA'] for s in splits]

x = np.arange(len(splits))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, normal_counts, width, label='NORMAL')
ax.bar(x + width/2, pneumonia_counts, width, label='PNEUMONIA')

ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel('枚数')
ax.set_title('クラス分布')
ax.legend()
plt.show()

### 画像カラーと画像サイズ(最大/最小)

In [ ]:
# グレースケールかRGBか、画像サイズのばらつき
sizes = []
modes = set()

for fname in os.listdir(os.path.join(train_dir, 'NORMAL'))[:50]:
    img = Image.open(os.path.join(train_dir, 'NORMAL', fname))
    sizes.append(img.size)
    modes.add(img.mode)

print(f'モード（RGB/L）: {modes}')
print(f'最小サイズ: {min(sizes)}')
print(f'最大サイズ: {max(sizes)}')

### 画素値の分布

In [ ]:
# 画素値の分布
transform = transforms.ToTensor()
values = []

for fname in os.listdir(os.path.join(train_dir, 'NORMAL'))[:50]:
    img = Image.open(os.path.join(train_dir, 'NORMAL', fname)).convert('RGB')
    values.append(transform(img).mean().item())

print(f'画素値の平均（0-1スケール）: {np.mean(values):.3f}')
print(f'標準偏差: {np.std(values):.3f}')

### 画素値ヒストグラム

In [ ]:
# 200枚分の画像の画素値の分布(ヒストグラム)
# これをもとに後のtransform.Normalizeでの正規化の値が決まる。(tenh活性化関数を使う場合のみ)
# 以下の結果は極端な偏りはないので、mean,stdともに0.5に正規化して問題なさそう。
# ReLU関数なら上記の正規化処理は不要
transform_for_check = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

dataset_for_check = datasets.ImageFolder(train_dir, transform=transform_for_check)

all_pixels = []
for i in range(200):
    img, _ = dataset_for_check[i]
    all_pixels.append(img.view(-1))

all_pixels = torch.cat(all_pixels)

plt.hist(all_pixels.numpy(), bins=50, range=(0, 1))
plt.title("Pixel Value Distribution (200 samples)")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.show()

### train画像データの表示

In [ ]:
# サンプル画像の見た目
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, label in enumerate(['NORMAL', 'PNEUMONIA']):
    files = os.listdir(os.path.join(train_dir, label))[:4]
    for j, fname in enumerate(files):
        # os.path.joinで画像の保存先～表示画像を指定
        img = Image.open(os.path.join(train_dir, label, fname)).convert('RGB')
        # 画像の配置を決定
        axes[i][j].imshow(img)
        axes[i][j].set_title(label)
        axes[i][j].axis('off')

plt.tight_layout()
plt.show()

### 破損画像の確認

In [ ]:
# 明らかな破損画像や読み込み失敗がないか
broken = []

for split, path in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    for label in ['NORMAL', 'PNEUMONIA']:
        for fname in os.listdir(os.path.join(path, label)):
            try:
                img = Image.open(os.path.join(path, label, fname))
                img.verify()
            except Exception:
                broken.append(f'{split}/{label}/{fname}')

print(f'破損画像数: {len(broken)}')
for b in broken:
    print(b)

### クラス不均衡の確認

In [ ]:
all_data = Path("./chest_xray")

splits = ["train", "val", "test"]
image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

summary = []

for split in splits:
    split_dir = all_data / split

    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            image_files = [
                f for f in class_dir.iterdir()
                if f.is_file() and f.suffix.lower() in image_extensions
            ]

            summary.append({
                "データ": split,
                "クラス": class_dir.name,
                "枚数": len(image_files)
            })

df = pd.DataFrame(summary)

df["割合"] = df.groupby("データ")["枚数"].transform(
    lambda x: x / x.sum()
)

df["割合"] = (df["割合"] * 100).round(2)

print(df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for i, split in enumerate(df["データ"].unique()):

    subset = df[df["データ"] == split]

    axes[i].bar(subset["クラス"], subset["枚数"])

    axes[i].set_title(split)
    axes[i].set_xlabel("Class")
    axes[i].set_ylabel("Count")

    for j, v in enumerate(subset["枚数"]):
        axes[i].text(j, v, str(v), ha="center")

plt.tight_layout()
plt.show()

### Augmentation テスト

In [ ]:
class MyImageDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']:
            self.files.extend(Path(root).rglob(ext))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        return Image.open(self.files[idx]).convert('RGB'), None

x_ray = MyImageDataset(DATA_DIR)

fig = plt.figure(figsize=(16, 8.5))
ops = [
    ('Rotate 10 degree', lambda im: F.rotate(im, angle=10)),
    ('Flip (horizontal)', lambda im: F.hflip(im)),
    ('Adjust contrast', lambda im: F.adjust_contrast(im, 2)),
    ('Adjust brightness', lambda im: F.adjust_brightness(im, 1.3)),
    ('Shift by 5%', lambda im: F.affine(im, angle=0,
        translate=(int(im.size[0]*0.05), int(im.size[1]*0.05)), scale=1.0, shear=0)),
    ('Scale up by 1.1x', lambda im: F.affine(im, angle=0, translate=(0, 0), scale=1.1, shear=0)),
]
for col, (title, fn) in enumerate(ops):
    img, _ = x_ray[col]
    ax = fig.add_subplot(2, 6, col + 1)
    ax.set_title(title, size=13); ax.imshow(img); ax.axis('off')
    ax = fig.add_subplot(2, 6, col + 7)
    ax.imshow(fn(img)); ax.axis('off')
plt.tight_layout()
plt.show()

## 前処理・DataLoader（src/dataset.py）

`make_transforms` / `build_loaders` は src に分割済み。Baseline は 1ch、転移学習は 3ch を使う。

In [ ]:
loaders_1ch = build_loaders(DATA_DIR, channels=1)
loaders_3ch = build_loaders(DATA_DIR, channels=3)
class_names = loaders_1ch[3].classes
print("classes:", class_names)
print("train/val/test:",
      len(loaders_1ch[0].dataset), len(loaders_1ch[1].dataset), len(loaders_1ch[2].dataset))

## モデルの学習・評価

4モデルを学習し、各結果を `results` に追記して最後に比較表にする。
比較行を作る補助関数 `make_row` を用意（推論速度・パラメータ数・モデルサイズも記録）。

In [ ]:
results = []

def make_row(name, res, model, test_dl):
    infer = measure_speed(model, test_dl, device)
    n_params, size_mb = model_stats(model)
    return {
        "model": name,
        "accuracy": round(res["accuracy"], 4),
        "precision": round(res["precision"], 4),
        "recall": round(res["recall"], 4),
        "f1": round(res["f1"], 4),
        "infer_ms_per_img": round(infer, 2),
        "params_M": round(n_params / 1e6, 2),
        "size_MB": round(size_mb, 2),
    }

### ① Baseline CNN（1ch・スクラッチ学習）

In [ ]:
train_dl, val_dl, test_dl, test_data = loaders_1ch

torch.manual_seed(1)
m_base = build_baseline().to(device)
opt = torch.optim.Adam(m_base.parameters(), lr=1e-4)
hist = train(m_base, opt, nn.CrossEntropyLoss(), 10, train_dl, val_dl, device)
plot_history(hist, "Baseline")
res_base = evaluate(m_base, test_dl, device, class_names, "Baseline")
results.append(make_row("Baseline", res_base, m_base, test_dl))

### ② ResNet18：2段階学習（Frozen → Fine-tuned）

まず特徴抽出部を **freeze して最終層(fc)だけ学習**（= Frozen）。  
その後 **後半層(layer3, layer4)を unfreeze して小さい学習率で fine-tune**（= Fine-tuned）。  
それぞれの時点でテスト評価し、比較表に2行として記録する。

In [ ]:
train_dl, val_dl, test_dl, test_data = loaders_3ch

torch.manual_seed(1)
m_res = build_resnet18().to(device)

# --- Phase 1: 特徴抽出 (backbone を freeze, fc だけ学習) ---
freeze_backbone_resnet(m_res)
opt = torch.optim.Adam(trainable_params(m_res), lr=1e-3)
hist1 = train(m_res, opt, nn.CrossEntropyLoss(), 5, train_dl, val_dl, device)
plot_history(hist1, "ResNet Frozen")
res_frozen = evaluate(m_res, test_dl, device, class_names, "ResNet Frozen")
results.append(make_row("ResNet Frozen", res_frozen, m_res, test_dl))

In [ ]:
# --- Phase 2: 後半層を unfreeze して fine-tune (小さい lr) ---
unfreeze_resnet_later(m_res, layers=("layer3", "layer4"))
opt = torch.optim.Adam(trainable_params(m_res), lr=1e-4)
hist2 = train(m_res, opt, nn.CrossEntropyLoss(), 5, train_dl, val_dl, device)
plot_history(hist2, "ResNet Fine-tuned")
res_ft = evaluate(m_res, test_dl, device, class_names, "ResNet Fine-tuned")
results.append(make_row("ResNet Fine-tuned", res_ft, m_res, test_dl))

### ③ EfficientNet-B0（フルファインチューニング）

In [ ]:
train_dl, val_dl, test_dl, test_data = loaders_3ch

torch.manual_seed(1)
m_eff = build_efficientnet_b0().to(device)
opt = torch.optim.Adam(m_eff.parameters(), lr=1e-4)
hist = train(m_eff, opt, nn.CrossEntropyLoss(), 10, train_dl, val_dl, device)
plot_history(hist, "EfficientNet-B0")
res_eff = evaluate(m_eff, test_dl, device, class_names, "EfficientNet-B0")
results.append(make_row("EfficientNet-B0", res_eff, m_eff, test_dl))

## モデル比較表（4モデル）

In [ ]:
import os
df = pd.DataFrame(results)
os.makedirs("outputs/metrics", exist_ok=True)
df.to_csv("outputs/metrics/comparison.csv", index=False)
print("保存: outputs/metrics/comparison.csv")
df

## 誤分類分析

代表として **ResNet Fine-tuned** の結果(`res_ft`)を使う(`test_data` は 3ch のもの)。

1. **方向別**: 見逃し(True=PNEUMONIA→Pred=NORMAL) と 過検出(True=NORMAL→Pred=PNEUMONIA)
2. **確信度別**: 高確率なのに誤った例 / 確率が低く迷っている例

In [ ]:
# 3ch の test_data を使う
_, _, test_dl_3, test_data_3 = loaders_3ch

# 1. 方向別 (見逃し / 過検出)
show_misclassified_by_type(test_data_3, res_ft, class_names, n=6)

In [ ]:
# 2. 確信度別 (確信していたのに誤り / 迷っている)
show_confidence_cases(test_data_3, res_ft, class_names, n=6)

## Grad-CAM：モデルが見ている箇所の可視化

ResNet Fine-tuned が画像のどこに着目して判断したかをヒートマップで表示する。
正しく分類できた肺炎画像と、誤分類した画像を並べると、着目領域の違いが分かりやすい。

In [ ]:
import numpy as np

y_true = res_ft["y_true"]; y_pred = res_ft["y_pred"]

# 正しく当てた肺炎(TP)を2枚、誤分類を2枚 選ぶ
tp = np.where((y_true == 1) & (y_pred == 1))[0][:2]
wrong = np.where(y_true != y_pred)[0][:2]
sample_idx = list(tp) + list(wrong)
print("表示する test インデックス:", sample_idx)

show_gradcam(m_res, test_data_3, sample_idx, device, class_names)